# N4 · Capstone: 升级版 harness 全家桶 + 产出研究 idea 卡 (配套 L14)

把 provider + compaction + long_horizon/hook + otel + eval **串成一个 harness**, 跑通跨窗口长任务,
然后把工程**落成研究**: 用 `critical-reading-gap` 的 idea 卡模板, 把 L13 的 harness gap 写成卡。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"
sys.path.insert(0, str(SRC))
import provider, compaction, long_horizon, otel_trace, harness_eval
print("src 组件就绪 (默认 MockProvider, 无需 API key)")

src 组件就绪 (默认 MockProvider, 无需 API key)


### 1) 装配并跑通一个跨窗口长任务 (全程可观测)

In [2]:
import tempfile
from long_horizon import run_long_horizon, demo_setup
from compaction import Compactor
from otel_trace import Tracer

work = tempfile.mkdtemp()
prov, goal, tools, store, goal_met = demo_setup(work, total_steps=8, early_stop_at=3)
tracer = Tracer()
res = run_long_horizon(prov, goal, tools, store, goal_met,
                       compactor=Compactor(600), tracer=tracer, max_windows=8, hook=True)
print("成功:", res.success, "| 窗口:", res.n_windows, "| 步:", res.total_steps,
      "| 累计上下文 tokens:", res.context_tokens_total)
print("span 树:\n" + tracer.render())

成功: True | 窗口: 2 | 步: 8 | 累计上下文 tokens: 2072
span 树:
▭ window-0 (Δ13)
  ◆ reason@w0.s1 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w0.s2 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w0.s3 (Δ3)
    → tool:do_step (Δ1)
▭ window-1 (Δ21)
  ◆ reason@w1.s1 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w1.s2 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w1.s3 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w1.s4 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w1.s5 (Δ3)
    → tool:do_step (Δ1)


### 2) 评测: 这个配置 vs 朴素配置

In [3]:
import pandas as pd
from harness_eval import evaluate, default_configs
df = pd.DataFrame(evaluate(default_configs(), tempfile.mkdtemp(), total_steps=8, early_stop_at=3))
print(df[["harness","success","context_tokens"]].to_string(index=False))

          harness  success  context_tokens
          A_naive    False             978
      B_hook_only     True            4168
C_hook_compaction     True            2072


### 3) 工程 → 研究: 把 L13 的 harness gap 写成 idea 卡 (复用 critical-reading-gap 模板)

In [4]:
from pathlib import Path
# 定位 critical-reading-gap 的 idea 卡模板 (跨专题复用)
root = Path.cwd()
while root.name and not (root / "learning").exists() and root.parent != root:
    root = root.parent
tpl = root / "learning" / "critical-reading-gap" / "templates" / "idea-card.md"
out = Path.cwd() / "_capstone_output"; out.mkdir(exist_ok=True)

print("idea 卡模板存在:", tpl.exists())
if tpl.exists():
    base = tpl.read_text(encoding="utf-8")
    # 起两张针对 harness 的 idea 卡 (来自 L13 的 G9/G10)
    for tag, gap in [("G9-repro-variance", "复现一篇热门 agent 论文并公开真实方差"),
                     ("G10-model-vs-harness", "把 SOTA agent 提升解耦成 模型贡献 vs harness 贡献")]:
        (out / f"IDEA-harness-{tag}.md").write_text(
            base + f"\n\n<!-- 预填: 来自 harness-engineering L13, gap={gap} -->\n", encoding="utf-8")
        print("  已起 idea 卡:", f"IDEA-harness-{tag}.md")
print("\n下一步: 打开这两张卡填满 (可证伪假设 + 最小验证实验 + 三筛), 带去找导师。")

idea 卡模板存在: True
  已起 idea 卡: IDEA-harness-G9-repro-variance.md
  已起 idea 卡: IDEA-harness-G10-model-vs-harness.md

下一步: 打开这两张卡填满 (可证伪假设 + 最小验证实验 + 三筛), 带去找导师。


## Capstone 收口

你刚刚:
1. 把 5 个 src 组件**装配成一个可观测、可评测的升级版 harness**, 跑通了跨窗口长任务;
2. 用对照实验证明了 harness 配置改变成败与成本;
3. **把这门工程课落成了 2 张研究 idea 卡** —— 工程 → 研究的桥, 在这里走完。

> 这是你 portfolio 里第一个「工程 ⨯ 研究」双栖专题。下一步可接 Module 9.4 `experiment-design`,
> 把 idea 卡里的「最小验证实验」做严谨。